<a href="https://colab.research.google.com/github/AdriBarrios96/Challenge3_TelecomX_II_DataScience/blob/main/Challenge3_Telecomx2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PROYECTO: Telecom X – Parte 2: Predicción de Cancelación (Churn)


In [10]:
import pandas as pd

#Cargamos el archivo CSV que limpiamos en la Parte 1
df = pd.read_csv('datos_tratados.csv')

display(df.head())

#Reordamos la informacion principal
print("\nInformación del DataFrame:")
df.info()

,customerID,Churn,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Charges.Monthly,Charges.Total,Cuentas_Diarias
0,0002-ORFBO,No,Female,0,Yes,Yes,9,Yes,No,DSL,...,No,Yes,Yes,No,One year,Yes,Mailed check,65.6,593.30,2.19
1,0003-MKNFE,No,Male,0,No,No,9,Yes,Yes,DSL,...,No,No,No,Yes,Month-to-month,No,Mailed check,59.9,542.40,2.00
2,0004-TLHLJ,Yes,Male,0,No,No,4,Yes,No,Fiber optic,...,Yes,No,No,No,Month-to-month,Yes,Electronic check,73.9,280.85,2.46
3,0011-IGKFF,Yes,Male,1,Yes,No,13,Yes,No,Fiber optic,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,98.0,1237.85,3.27
4,0013-EXCHZ,Yes,Female,1,Yes,No,3,Yes,No,Fiber optic,...,No,Yes,Yes,No,Month-to-month,Yes,Mailed check,83.9,267.40,2.80



Información del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7267 non-null   object 
 1   Churn             7043 non-null   object 
 2   gender            7267 non-null   object 
 3   SeniorCitizen     7267 non-null   int64  
 4   Partner           7267 non-null   object 
 5   Dependents        7267 non-null   object 
 6   tenure            7267 non-null   int64  
 7   PhoneService      7267 non-null   object 
 8   MultipleLines     7267 non-null   object 
 9   InternetService   7267 non-null   object 
 10  OnlineSecurity    7267 non-null   object 
 11  OnlineBackup      7267 non-null   object 
 12  DeviceProtection  7267 non-null   object 
 13  TechSupport       7267 non-null   object 
 14  StreamingTV       7267 non-null   object 
 15  StreamingMovies   7267 non-null   object 
 16  Contract      

##🛠️ Preparación de los Datos

In [11]:
#Eliminamos la columna 'customerID' ya que no aporta valor predictivo
df = df.drop(columns=['customerID'])

#Visualizamos el DataFrame
print(f"Tenemos {len(df.columns)} columnas listas.")
print("Columnas:\n", df.columns.tolist())

Tenemos 21 columnas listas.
Columnas:
 ['Churn', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Charges.Monthly', 'Charges.Total', 'Cuentas_Diarias']


In [12]:
#Eliminamos las filas nulas
df = df.dropna(subset=['Churn'])
print(f"Filas sin nulos: {len(df)}")

Filas sin nulos: 7043


####Transformación de Variables (One-Hot Encoding)

In [13]:
#Convertimos 'Churn' en 1 (Yes) y 0 (No) para predecir mejor.
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

#Aplicamos One-Hot Encoding y drop_first=True para Regresión Lineal
df_numerico = pd.get_dummies(df, drop_first=True)

#Pasamos a entero (int) para mayor compatibilidad
df_numerico = df_numerico.astype(int)

#Visualizamos el resultado
print("Exito al transformar...")
print(f"Pasamos de {len(df.columns)} columnas a {len(df_numerico.columns)} columnas numericas.")
display(df_numerico.head())

Exito al transformar...
Pasamos de 21 columnas a 32 columnas numericas.


,Churn,SeniorCitizen,tenure,Charges.Monthly,Charges.Total,Cuentas_Diarias,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,9,65,593,2,0,1,1,1,...,0,1,0,0,1,0,1,0,0,1
1,0,0,9,59,542,2,1,0,0,1,...,0,0,0,1,0,0,0,0,0,1
2,1,0,4,73,280,2,1,0,0,1,...,0,0,0,0,0,0,1,0,1,0
3,1,1,13,98,1237,3,1,1,0,1,...,0,1,0,1,0,0,1,0,1,0
4,1,1,3,83,267,2,0,1,0,1,...,0,1,0,0,0,0,1,0,0,1


####Análisis de Balance de Clases

In [14]:
# alculamos las cantidades
conteo_clases = df_numerico['Churn'].value_counts()

#Calculamos los porcentajes
porcentajes_clases = df_numerico['Churn'].value_counts(normalize=True) * 100

print("- - BALANCE DE CHURN - -\n")

print("Cantidades absolutas:")
print(f"Se quedaron (0): {conteo_clases[0]} clientes")
print(f"Se fueron (1): {conteo_clases[1]} clientes\n")

print("Proporciones (%):")
print(f"Se quedaron (0): {porcentajes_clases[0]:.2f}%")
print(f"Se fueron (1): {porcentajes_clases[1]:.2f}%")

- - BALANCE DE CHURN - -

Cantidades absolutas:
Se quedaron (0): 5174 clientes
Se fueron (1): 1869 clientes

Proporciones (%):
Se quedaron (0): 73.46%
Se fueron (1): 26.54%


####Utilizando SMOTE

In [15]:
# Importamos SMOTE
from imblearn.over_sampling import SMOTE

X = df_numerico.drop('Churn', axis=1)
y = df_numerico['Churn']

#Usamos random_state=42 para que siempre de el mismo resultado
smote = SMOTE(random_state=42)

#Creamos los datos
X_balanceado, y_balanceado = smote.fit_resample(X, y)

#Visualizamos el resultado
print("- - NUEVO BALANCE DE CHURN - -\n")
print("Cantidades absolutas:")
print(y_balanceado.value_counts())

print("\nProporciones (%):")
print(y_balanceado.value_counts(normalize=True) * 100)

print(f"\nAhora tenemos {len(X_balanceado)} filas en total para nuestro modelo de forma justa.")

- - NUEVO BALANCE DE CHURN - -

Cantidades absolutas:
Churn
0    5174
1    5174
Name: count, dtype: int64

Proporciones (%):
Churn
0    50.0
1    50.0
Name: proportion, dtype: float64

Ahora tenemos 10348 filas en total para nuestro modelo de forma justa.


####Normalizacion y estandarizacion

In [16]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

scaler = StandardScaler()

X_escalado = scaler.fit_transform(X_balanceado)

#Volvemo a armar el DataFrame
X_balanceado_scaled = pd.DataFrame(X_escalado, columns=X_balanceado.columns)

#Visualizacion de resultados
columnas_numericas_originales = ['tenure', 'Charges.Monthly', 'Charges.Total', 'Cuentas_Diarias']
print("Asi quedaron las nuevas variables numericas:")
display(X_balanceado_scaled[columnas_numericas_originales].head())

Asi quedaron las nuevas variables numericas:


,tenure,Charges.Monthly,Charges.Total,Cuentas_Diarias
0,-0.777844,-0.084582,-0.664483,0.291950
1,-0.777844,-0.293824,-0.687803,0.291950
2,-0.986309,0.194406,-0.807603,0.291950
3,-0.611072,1.066245,-0.370012,1.251334
4,-1.028002,0.543142,-0.813548,0.291950


##🎯 Correlación y Selección de Variables